# HW2

In [8]:
import numpy as np
import pandas as pd
import re
import os
from sklearn.pipeline import Pipeline
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score # для разделения с учетом групп
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
import random
from collections import defaultdict
from solution_utils_old import(MyModelReactions, read_dialogs, read_labels, get_groups, clean_normalize_dialog, calculate_metrics, save_model, SEED)


In [2]:
DATA_PATH = "../../hw1/hw1_train.jsonl"
LABELS = np.array(["positive", "negative", "neutral"])

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))
reactions = read_labels(DATA_PATH, "reaction")
labeled_dialogs = [d for d in dialogs if reactions.get(d.get('dialog_id')) is not None]
print("размеченных диалогов", len(labeled_dialogs))

диалогов: 10000
размеченных диалогов 3000


In [3]:
user_texts = [clean_normalize_dialog(d) for d in labeled_dialogs]
n_gr, groups_ids = get_groups(user_texts)
groups_ids = np.array(groups_ids)
gkf = GroupKFold(n_splits=5)
print(f"групп: {n_gr}")
sizes = np.bincount(groups_ids)
print(f"групп >1: {(sizes > 1).sum()}")
print(f"макс размер: {sizes.max()}")
print(f"диалогов в группах >1: {sizes[sizes > 1].sum()}")

групп: 2978
групп >1: 17
макс размер: 4
диалогов в группах >1: 39


In [4]:
X = np.array(user_texts)
y = np.array([reactions[d["dialog_id"]] for d in labeled_dialogs])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_val, idx_test = next(gss.split(X, y, groups_ids))
gss = GroupShuffleSplit(n_splits=1, train_size=0.8125, random_state=SEED)
idx_train, idx_val = next(gss.split(X[idx_train_val], y[idx_train_val], groups_ids[idx_train_val]))

X_train, y_train = X[idx_train_val[idx_train]], y[idx_train_val[idx_train]] # 65%
X_val, y_val = X[idx_train_val[idx_val]], y[idx_train_val[idx_val]] # 15%
X_test, y_test = X[idx_test], y[idx_test] # 20%

groups_ids_train = groups_ids[idx_train_val[idx_train]]

In [5]:
for name, labels_part in [
    ("train", y_train),
    ("validation", y_val),
    ("test", y_test),
]:
    print(f"\n{name}: {len(labels_part)} примеров")
    print(
        pd.Series(labels_part)
        .value_counts(normalize=True)
        .reindex(LABELS, fill_value=0)
        .round(3)
    )


train: 1954 примеров
positive    0.100
negative    0.317
neutral     0.583
Name: proportion, dtype: float64

validation: 449 примеров
positive    0.114
negative    0.303
neutral     0.584
Name: proportion, dtype: float64

test: 597 примеров
positive    0.094
negative    0.281
neutral     0.625
Name: proportion, dtype: float64


Для деревьев надо выбрать другой TfIdf, с нашей разреженной гигантской матрицей деревья будут обучаться очень долго. Попробуем на кросс-валидации примерно подобрать хороший tfidf.

In [13]:
variants = {
    "full":        TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True),
    "max3000":     TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=3000),
    "max5000":     TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=5000),
    "mindf3":      TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, min_df=3),
    "mindf5":      TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, min_df=5),
}

bag = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=10, min_samples_leaf=3, random_state=SEED),
                        n_estimators=100, oob_score=True, random_state=SEED)
forest = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=3, oob_score=True, random_state=SEED)

rows = []
for name, vec in variants.items():
    Xtr = vec.fit_transform(X_train)
    scores_bag = cross_val_score(bag, Xtr, y_train, cv=gkf, groups=groups_ids_train, scoring='f1_macro')
    scores_forest = cross_val_score(forest, Xtr, y_train, cv=gkf, groups=groups_ids_train ,scoring='f1_macro')
    rows.append({
        "vectorizer": name,
        "bag_mean": round(scores_bag.mean(), 4),
        "rf_mean":  round(scores_forest.mean(), 4),
    })

df = pd.DataFrame(rows)
df["best"] = df[["bag_mean", "rf_mean"]].max(axis=1)
df = df.sort_values("best", ascending=False).drop(columns="best").reset_index(drop=True)
df

,vectorizer,bag_mean,rf_mean
0,max3000,0.5521,0.3621
1,mindf5,0.5454,0.3406
2,full,0.5452,0.2532
3,mindf3,0.5442,0.3169
4,max5000,0.5379,0.3421


Выбрали определенную вариацию tfidf. Очень маленький f1 для леса получился, но с этим разберемся отдельно.

In [14]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # добавляем биграммы
    sublinear_tf=True,     # уменьшает скорость роста влияния слова от его частоты
    max_features=3000,     # ограничимся 3000 самыми частыми словами и биграммами
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

## Бэггинг

In [28]:
param_grid = {
    'n_estimators': [50, 100],
    'estimator__max_depth': [5, 10, None],
    'estimator__min_samples_leaf': [1, 3, 5]
}

grid_bag = GridSearchCV(
    BaggingClassifier(estimator=DecisionTreeClassifier(random_state=SEED),
                      oob_score=True, random_state=SEED),
    param_grid,
    cv=gkf,
    scoring='f1_macro',
    n_jobs=-1
)
grid_bag.fit(X_train_tfidf, y_train, groups=groups_ids_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",BaggingClassi...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'estimator__max_depth': [5, 10, ...], 'estimator__min_samples_leaf': [1, 3, ...], 'n_estimators': [50, 100]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",GroupKFold(n_...shuffle=False)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable 

In [29]:
results = pd.DataFrame(grid_bag.cv_results_)[[
    'param_n_estimators',
    'param_estimator__max_depth',
    'param_estimator__min_samples_leaf',
    'mean_test_score'
]].copy()

results.columns = [
    'n_estimators', 'max_depth', 'min_samples_leaf',
    'f1']

results = results.round(4).sort_values('f1', ascending=False)
print(f"Best params: {grid_bag.best_params_}")
print(f"OOB F1_macro: {grid_bag.best_estimator_.oob_score_:.4f}") # оценка вне выборки для лучших параметров
results

Best params: {'estimator__max_depth': None, 'estimator__min_samples_leaf': 3, 'n_estimators': 50}
OOB F1_macro: 0.6377


,n_estimators,max_depth,min_samples_leaf,f1
14,50,None,3,0.5695
13,100,None,1,0.5667
12,50,None,1,0.5658
15,100,None,3,0.5648
17,100,None,5,0.5596
16,50,None,5,0.5546
8,50,10,3,0.5536
9,100,10,3,0.5521
7,100,10,1,0.5515
11,100,10,5,0.5501


Лучшие параметры: max_depth = None, min_samples_leaf = 3, n_estimators = 50

In [30]:
bag_n_estinators = 50
bag_max_depth = None
bag_min_s_leaf = 3
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=bag_max_depth, min_samples_leaf=bag_min_s_leaf, random_state=SEED),
    oob_score=True, random_state=SEED
).fit(X_train_tfidf, y_train)
print(f"OOB: {bagging.oob_score_}")
y_pred_bag = bagging.predict(X_val_tfidf)
print(calculate_metrics(y_val, y_pred_bag, LABELS))

OOB: 0.5951893551688844
{'accuracy': 0.6659242761692651, 'balanced_accuracy': 0.5394339669710123, 'macro-f1': 0.5685425055101375, 'weighted-f1': 0.6462519301195443, 'confusion_matrix': array([[ 17,   7,  27],
       [  2,  59,  75],
       [  7,  32, 223]]), 'per_class_metrics': {'precision_positive': np.float64(0.6538461538461539), 'recall_positive': np.float64(0.3333333333333333), 'f1_positive': np.float64(0.44155844155844154), 'precision_negative': np.float64(0.6020408163265306), 'recall_negative': np.float64(0.4338235294117647), 'f1_negative': np.float64(0.5042735042735043), 'precision_neutral': np.float64(0.6861538461538461), 'recall_neutral': np.float64(0.851145038167939), 'f1_neutral': np.float64(0.7597955706984668)}}


c:\Users\admin\Desktop\ML\venvml\Lib\site-packages\sklearn\ensemble\_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
c:\Users\admin\Desktop\ML\venvml\Lib\site-packages\sklearn\ensemble\_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


## Случайный лес

По умолчанию в качестве max_features берется корень, для нашего ограниченного количества признаков это очень мало, попробуем разное.

In [25]:
param_grid = {
    'max_features': ['sqrt', 'log2', 500, 1000],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 3, 5],
}

grid_forest = GridSearchCV(
    RandomForestClassifier(n_estimators=50, oob_score=True, random_state=SEED),
    param_grid,
    cv=gkf,
    scoring='f1_macro',
    n_jobs=-1 # быстрее
)
grid_forest.fit(X_train_tfidf, y_train, groups=groups_ids_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [10, 20, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",GroupKFold(n_...shuffle=False)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",T

In [26]:
results = pd.DataFrame(grid_forest.cv_results_)[[
    'param_max_features',
    'param_max_depth',
    'param_min_samples_leaf',
    'mean_test_score'
]].copy()

results.columns = [
    'max_features', 'max_depth', 'min_samples_leaf', 'f1']

results = results.round(4).sort_values('f1', ascending=False)
print(f"Best params: {grid_forest.best_params_}")
print(f"OOB F1_macro: {grid_forest.best_estimator_.oob_score_:.4f}") # оценка вне выборки для лучших параметров
results

Best params: {'max_depth': None, 'max_features': 500, 'min_samples_leaf': 1}
OOB F1_macro: 0.6418


,max_features,max_depth,min_samples_leaf,f1
30,500,None,1,0.5744
31,500,None,3,0.5705
34,1000,None,3,0.5650
35,1000,None,5,0.5628
18,500,20,1,0.5602
32,500,None,5,0.5602
33,1000,None,1,0.5595
20,500,20,5,0.5567
19,500,20,3,0.5547
22,1000,20,3,0.5537


Предположение подтвердилось, еще при подборе tfidf проблема была в слишком маленькой выборе параметров для отдельных дерефьев леса.

In [31]:
forest_max_features = 500
forest_max_depth = None
forest_min_s_leaf = 1
forest = RandomForestClassifier(n_estimators=50, max_features=forest_max_features, max_depth=forest_max_depth,
                       min_samples_leaf=forest_min_s_leaf, oob_score=True, random_state=SEED).fit(X_train_tfidf, y_train)
print(f"OOB: {forest.oob_score_}")
y_pred_forest = forest.predict(X_val_tfidf)
print(calculate_metrics(y_val, y_pred_forest, LABELS))

OOB: 0.6417604912998977
{'accuracy': 0.6636971046770601, 'balanced_accuracy': 0.5352554507808213, 'macro-f1': 0.5643825042881647, 'weighted-f1': 0.6448284670463936, 'confusion_matrix': array([[ 16,   8,  27],
       [  2,  61,  73],
       [  6,  35, 221]]), 'per_class_metrics': {'precision_positive': np.float64(0.6666666666666666), 'recall_positive': np.float64(0.3137254901960784), 'f1_positive': np.float64(0.4266666666666667), 'precision_negative': np.float64(0.5865384615384616), 'recall_negative': np.float64(0.4485294117647059), 'f1_negative': np.float64(0.5083333333333333), 'precision_neutral': np.float64(0.6884735202492211), 'recall_neutral': np.float64(0.8435114503816794), 'f1_neutral': np.float64(0.758147512864494)}}


## Cтекинг